## Railway Dataset (2012 - 2022)

- This dataset contains the state-wise figures of railway track length and route length from year 2012-2022. 
- Each row represent **financial year**, **state_name**, **state_code**, **route_length**, **track_length**
- Key for the dataset is **[state_name, year]**

## Domain specific check

- If in the dataset we get **track_length** < **Route_length** then the data is invalid. 

In [33]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [34]:
df = pd.read_csv(r"C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks.csv")

In [35]:
print("Shape\n", df.shape)

Shape
 (307, 6)


In [36]:
print("Data Types\n", df.dtypes)

Data Types
 id               int64
year            object
state_code       int64
state_name      object
route_length     int64
total_track      int64
dtype: object


- **Year** is of string data type. 

In [37]:
df.head(20)

,id,year,state_code,state_name,route_length,total_track
0,0,2012-2013,28,Arunachal Pradesh,5322,9941
1,1,2012-2013,12,Assam,1,3
2,2,2012-2013,18,Bihar,2459,3525
3,3,2012-2013,10,Chhatisgarh,3656,6595
4,4,2012-2013,22,Delhi,1196,2625
5,5,2012-2013,7,Goa,183,699
6,6,2012-2013,30,Gujarat,69,98
7,7,2012-2013,24,Haryana,5257,7388
8,8,2012-2013,6,Himachal Pradesh,1630,2973
9,9,2012-2013,2,Jammu and Kashmir,296,357


In [38]:
print("Missing Values\n", df.isna().sum())

Missing Values
 id              0
year            0
state_code      0
state_name      0
route_length    0
total_track     0
dtype: int64


- Dataset has no missing values. 

In [39]:
print("Duplicate Rows\n", df.duplicated().sum())

Duplicate Rows
 0


- Dataset contains no duplicated rows
- Since 1 row represents, one state per year so key becomes (state_name, year)

In [40]:
key = ['state_name', 'year']
print("Duplicate Rows based on key\n", df.duplicated(subset=key).sum())

Duplicate Rows based on key
 0


- No duplicated rows based on the Key
- Using **Describe** to inspect if the route_length and track_length contains any invalid values. 

In [41]:
print(df[["route_length", "total_track"]].describe())
# No invalid values or outliers observed in the dataset. The values are within the expected range.

       route_length   total_track
count    307.000000    307.000000
mean    2186.609121   3972.934853
std     2317.682900   4143.963374
min        1.000000      3.000000
25%       69.000000    117.500000
50%     1703.000000   3058.000000
75%     3817.000000   6835.000000
max    10324.000000  16063.000000


In [42]:
print("\nUnique state_code :", df["state_code"].nunique())
print("Unique state_name :", df["state_name"].nunique())


Unique state_code : 31
Unique state_name : 31


#### Domain Specific Data validation check

In [43]:
# Check for rows where route_length is greater than total_track (which is not possible)
violations =  df[df["route_length"] > df['total_track']]
print('Rows where route_length is greater than total_track (Not possible):', len(violations))
print(violations)

Rows where route_length is greater than total_track (Not possible): 1
      id       year  state_code   state_name  route_length  total_track
178  178  2017-2018           5  Uttarakhand           341          189


- Uttarakhand State has a invalid data for **year 2017-18**

In [44]:
# Inspecting the data for the state of Uttarakhand
print(df[df['state_name'] == 'Uttarakhand'][['year', 'route_length', 'total_track']].to_string(index=False))

     year  route_length  total_track
2012-2013           345          556
2013-2014           345          518
2014-2015           345          518
2015-2016           340          509
2016-2017           340          465
2017-2018           341          189
2018-2019           341          527
2019-2020           346          528
2020-2021           346          555
2021-2022           346          517


### Handling of Invalid Data

In [45]:
# Calculating the estimate for the missing value of total_track for Uttarakhand in 2017-2018 using the average of the neighboring years (2016-2017 and 2018-2019)
uk = df[df["state_name"] == "Uttarakhand"]
before = uk.loc[uk["year"] == "2016-2017", "total_track"].values[0]
after  = uk.loc[uk["year"] == "2018-2019", "total_track"].values[0]
estimate = round((before + after) / 2)
print(f"Neighbors: {before} and {after}  ->  estimate = {estimate}")

Neighbors: 465 and 527  ->  estimate = 496


In [46]:
# Masking the row for Uttarakhand in 2017-2018 and updating the total_track value with the calculated estimate
mask = (df["state_name"] == "Uttarakhand") & (df["year"] == "2017-2018")
df.loc[mask, "total_track"] = estimate

In [47]:
print("Invariant violations remaining:", (df["route_length"] > df["total_track"]).sum())
print(df[mask][["year", "state_name", "route_length", "total_track"]].to_string(index=False))

Invariant violations remaining: 0
     year  state_name  route_length  total_track
2017-2018 Uttarakhand           341          496


- The invalid value for the state of Uttarakhand for the year 2017-18 has been be cleaned using statistical method since it appears in the between of series and seems valid approach to fill the estimated correct value using statistics. 

### State_Code vs State_name verification

- Need is to verify if the state_code vs state_name relation remains constant throughout the dataset

In [48]:
# Checking for states with multiple names for the same state_code
codes_with_multiple_names = df.groupby('state_name')['state_code'].nunique()
print("\nHow many names each codes maps to:\n" , codes_with_multiple_names[codes_with_multiple_names > 1])


How many names each codes maps to:
 state_name
Andhra Pradesh       2
Arunachal Pradesh    2
Assam                2
Bihar                2
Chhatisgarh          2
Delhi                2
Goa                  2
Gujarat              2
Haryana              2
Himachal Pradesh     2
Jammu and Kashmir    2
Jharkhand            2
Karnataka            2
Kerala               2
Madhya Pradesh       2
Name: state_code, dtype: int64


- Multiple state_name are mapping to different state_code within the dataset

In [49]:
# Creating a pivot table to check for state_name changes over the years for the same state_code
grid = df.pivot_table(index='state_name', columns='year', values='state_code', aggfunc='first')
changing = grid[grid.nunique(axis=1) > 1]

In [50]:
pd.set_option('display.width', 200, 'display.max_columns', 20)
print(changing)

year               2012-2013  2013-2014  2014-2015  2015-2016  2016-2017  2017-2018  2018-2019  2019-2020  2020-2021  2021-2022
state_name                                                                                                                     
Andhra Pradesh          23.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0
Arunachal Pradesh       28.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0
Assam                   12.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0
Bihar                   18.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0
Chhatisgarh             10.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0
Delhi                   22.0        7.0        7.0        7.0        7.0        7.0        7.0        7.

## Observation

- State code is stable from 2013 onwards
- State code after 2013 can be used as correct state codes for the dataset

In [51]:
# Creating a canonical map of state_name to state_code. 
good_year = df[df["year"] != "2012-2013"]
canonical_map = good_year.drop_duplicates("state_name").set_index("state_name")["state_code"].to_dict()
print("Canonical map size:", len(canonical_map), "states")

Canonical map size: 31 states


In [52]:
# Mapping the canonical state_code to the dataframe based on state_name
df["state_code"] = df["state_name"].map(canonical_map)

In [53]:
# Verifying the mapping
print("Codes mapping to >1 name:", (df.groupby('state_code')['state_name'].nunique() > 1).sum())
print("Rows that failed to map (NaN):", df["state_code"].isna().sum())

Codes mapping to >1 name: 0
Rows that failed to map (NaN): 0


- State code are corrected and now each state maps to 1 state code in the dataset

In [56]:
# Correcting the spelling of Chhattisgarh in the state_name column
df["state_name"] = df["state_name"].replace({"Chhatisgarh": "Chhattisgarh"})

In [55]:
out = r"C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks-clean.csv"
df.to_csv(out, index=False)
print("\nSaved cleaned data to:", out)


Saved cleaned data to: C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks-clean.csv
